# 06.04 — Rolling Features

Create historical rolling summaries after shifting the target by one hour.

In [1]:
# Import libraries
from pathlib import Path
import sys

In [2]:
# define the root directory of the project
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / 'configs' / 'feature_engineering.yaml'
CONFIG_PATH

WindowsPath('e:/jcuenca/OneDrive - GUSCanada/5toTerm/01_Capstone/Codigo/ontario-electricity-peak-risk/configs/feature_engineering.yaml')

In [3]:
# Import module to manage the Feature Engineering process
from src.ontario_peak_risk.feature_engineering.common import (
    load_feature_config,
    load_clean_dataset,
    ensure_directories,
)

In [4]:
# Config Feature Engineering process and Load the dataset
CONFIG, _ = load_feature_config(CONFIG_PATH)
REPORTS_DIR, DOCS_DIR = ensure_directories(CONFIG, PROJECT_ROOT)
clean_dataset = load_clean_dataset(CONFIG, PROJECT_ROOT)
clean_dataset.shape

(262944, 66)

In [5]:
# Import the functions to run baseline Feature Engineering process and add temporal features
from src.ontario_peak_risk.feature_engineering.base_features import build_base_dataset
from src.ontario_peak_risk.feature_engineering.temporal_features import add_temporal_features
from src.ontario_peak_risk.feature_engineering.lag_features import add_target_lags
from src.ontario_peak_risk.feature_engineering.rolling_features import add_target_rolling_features

In [6]:
# Run the baseline Feature Engineering process
base_dataset, _ = build_base_dataset(clean_dataset, CONFIG)
temporal_dataset, _ = add_temporal_features(base_dataset, CONFIG)
lag_dataset, _ = add_target_lags(temporal_dataset, CONFIG)
rolling_dataset, rolling_dictionary = add_target_rolling_features(lag_dataset, CONFIG)
rolling_dictionary

,feature_name,feature_family,source_columns,description,forecasting_candidate,peak_risk_candidate,computation_scope,leakage_risk
0,total_consumption_kwh_rolling_mean_3h,target_rolling,total_consumption_kwh,Mean of the previous 3 hourly consumption obse...,True,True,historical_only,low_if_shift_is_preserved
1,total_consumption_kwh_rolling_std_3h,target_rolling,total_consumption_kwh,Std of the previous 3 hourly consumption obser...,True,True,historical_only,low_if_shift_is_preserved
2,total_consumption_kwh_rolling_min_3h,target_rolling,total_consumption_kwh,Min of the previous 3 hourly consumption obser...,True,True,historical_only,low_if_shift_is_preserved
3,total_consumption_kwh_rolling_max_3h,target_rolling,total_consumption_kwh,Max of the previous 3 hourly consumption obser...,True,True,historical_only,low_if_shift_is_preserved
4,total_consumption_kwh_rolling_mean_6h,target_rolling,total_consumption_kwh,Mean of the previous 6 hourly consumption obse...,True,True,historical_only,low_if_shift_is_preserved
5,total_consumption_kwh_rolling_std_6h,target_rolling,total_consumption_kwh,Std of the previous 6 hourly consumption obser...,True,True,historical_only,low_if_shift_is_preserved
6,total_consumption_kwh_rolling_min_6h,target_rolling,total_consumption_kwh,Min of the previous 6 hourly consumption obser...,True,True,historical_only,low_if_shift_is_preserved
7,total_consumption_kwh_rolling_max_6h,target_rolling,total_consumption_kwh,Max of the previous 6 hourly consumption obser...,True,True,historical_only,low_if_shift_is_preserved
8,total_consumption_kwh_rolling_mean_24h,target_rolling,total_consumption_kwh,Mean of the previous 24 hourly consumption obs...,True,True,historical_only,low_if_shift_is_preserved
9,total_consumption_kwh_rolling_std_24h,target_rolling,total_consumption_kwh,Std of the previous 24 hourly consumption obse...,True,True,historical_only,low_if_shift_is_preserved


In [7]:
# Display the rolling dataset with the new features
rolling_columns = [c for c in rolling_dataset.columns if '_rolling_' in c and c.startswith('total_consumption_kwh')]
rolling_dataset[['fsa','timestamp','total_consumption_kwh',*rolling_columns]].head(30)

,fsa,timestamp,total_consumption_kwh,total_consumption_kwh_rolling_mean_3h,total_consumption_kwh_rolling_std_3h,total_consumption_kwh_rolling_min_3h,total_consumption_kwh_rolling_max_3h,total_consumption_kwh_rolling_mean_6h,total_consumption_kwh_rolling_std_6h,total_consumption_kwh_rolling_min_6h,total_consumption_kwh_rolling_max_6h,total_consumption_kwh_rolling_mean_24h,total_consumption_kwh_rolling_std_24h,total_consumption_kwh_rolling_min_24h,total_consumption_kwh_rolling_max_24h,total_consumption_kwh_rolling_mean_168h,total_consumption_kwh_rolling_std_168h,total_consumption_kwh_rolling_min_168h,total_consumption_kwh_rolling_max_168h
0,L4T,2021-01-01 00:00:00,10276.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,L4T,2021-01-01 01:00:00,9585.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,L4T,2021-01-01 02:00:00,9015.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,L4T,2021-01-01 03:00:00,8593.1,9625.733333,631.531158,9015.5,10276.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,L4T,2021-01-01 04:00:00,8353.5,9064.566667,497.816887,8593.1,9585.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,L4T,2021-01-01 05:00:00,8334.4,8654.033333,335.180031,8353.5,9015.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,L4T,2021-01-01 06:00:00,8499.5,8427.000000,144.163484,8334.4,8593.1,9026.366667,773.908500,8334.4,10276.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,L4T,2021-01-01 07:00:00,8784.2,8395.800000,90.313177,8334.4,8499.5,8730.183333,486.380413,8334.4,9585.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,L4T,2021-01-01 08:00:00,9313.2,8539.366667,227.534664,8334.4,8784.2,8596.700000,263.802206,8334.4,9015.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,L4T,2021-01-01 09:00:00,10062.2,8865.633333,412.917017,8499.5,9313.2,8646.316667,366.378479,8334.4,9313.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
